# Hypothesis tests for RQ0 and RQ1

The notebook follows the analysis plan in `docs.md`: two-factor models and assumption checks, baseline comparisons for RQ0, and absolute-deviation comparisons plus the time criterion for RQ1.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multitest import multipletests

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'warmup_cooldown' / 'experiments').exists()), Path.cwd())
OUT = ROOT / 'warmup_cooldown' / 'statistics'
df = pd.read_csv(OUT / 'cleaned_runs.csv')
df['duration_s'] = df['duration_s'].astype(int)
df['benchmark'] = df['benchmark'].astype(str)
DURATIONS = [5, 30, 60, 120, 210]
BASELINE, REFERENCE = 30, 210
ALPHA, RATIO_LIMIT = .05, 1.10

def bh(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    return multipletests(pvalues, alpha=ALPHA, method='fdr_bh')[1]

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    return (np.sign(x[:, None] - y[None, :]).sum() / (len(x) * len(y)))

def model_tables(value):
    model = ols(f'{value} ~ C(duration_s) * C(benchmark)', data=df).fit()
    anova = anova_lm(model, typ=2).reset_index(names='term')
    shapiro = stats.shapiro(model.resid)
    cells = [group[value].to_numpy() for _, group in df.groupby(['benchmark','duration_s'])]
    levene = stats.levene(*cells, center='median')
    assumptions = pd.DataFrame([
        {'measure': value, 'test': 'Shapiro-Wilk residual normality', 'statistic': shapiro.statistic, 'p_value': shapiro.pvalue},
        {'measure': value, 'test': 'Brown-Forsythe/Levene cell variance', 'statistic': levene.statistic, 'p_value': levene.pvalue},
    ])
    return anova, assumptions

rq0_anova, rq0_assumptions = model_tables('energy_j')
df['absolute_deviation_j'] = df.groupby(['benchmark','duration_s'])['energy_j'].transform(lambda x: (x - x.median()).abs())
rq1_anova, rq1_assumptions = model_tables('absolute_deviation_j')
pd.concat([rq0_anova.assign(outcome='energy_j'), rq1_anova.assign(outcome='absolute_deviation_j')]).to_csv(OUT / 'table_factorial_anova.csv', index=False)
pd.concat([rq0_assumptions, rq1_assumptions]).to_csv(OUT / 'table_assumption_checks.csv', index=False)
display(pd.concat([rq0_anova.assign(outcome='energy_j'), rq1_anova.assign(outcome='absolute_deviation_j')]).round(5))
display(pd.concat([rq0_assumptions, rq1_assumptions]).round(5))

Because both outcomes violate the factorial assumptions, the report-specified fallback is an aligned rank transform (ART) factorial analysis. The ART results below are the inferential factorial results; the Mann--Whitney tests later in the notebook are post-hoc baseline/reference comparisons.

In [ ]:
# Aligned Rank Transform (ART) factorial analysis.
# For each target effect, remove the nuisance effects, rank the aligned response,
# and fit the full factorial model to those ranks.
def art_factorial(value):
    work = df[['duration_s', 'benchmark', value]].copy()
    grand = work[value].mean()
    duration_mean = work.groupby('duration_s')[value].transform('mean')
    benchmark_mean = work.groupby('benchmark')[value].transform('mean')
    cell_mean = work.groupby(['duration_s', 'benchmark'])[value].transform('mean')
    aligned = {
        'duration': work[value] - benchmark_mean + grand,
        'benchmark': work[value] - duration_mean + grand,
        'duration:benchmark': work[value] - duration_mean - benchmark_mean + grand,
    }
    rows = []
    for effect, aligned_values in aligned.items():
        ranked = stats.rankdata(aligned_values, method='average')
        work['_aligned_rank'] = ranked
        model = ols('_aligned_rank ~ C(duration_s) * C(benchmark)', data=work).fit()
        table = anova_lm(model, typ=2).reset_index(names='term')
        target = {'duration': 'C(duration_s)',
                  'benchmark': 'C(benchmark)',
                  'duration:benchmark': 'C(duration_s):C(benchmark)'}[effect]
        result = table[table['term'].eq(target)].iloc[0]
        rows.append({'outcome': value, 'effect': effect,
                     'sum_sq': result['sum_sq'], 'df': result['df'],
                     'F': result['F'], 'p_value': result['PR(>F)']})
    return pd.DataFrame(rows)

art_results = pd.concat([
    art_factorial('energy_j'),
    art_factorial('absolute_deviation_j'),
], ignore_index=True)
art_results.to_csv(OUT / 'table_art_anova.csv', index=False)
display(art_results.round(5))

In [ ]:
# RQ0: two-sided comparisons with the 30 s baseline, corrected over all tests.
rows = []
for benchmark in sorted(df['benchmark'].unique()):
    base = df.query('benchmark == @benchmark and duration_s == @BASELINE')['energy_j'].to_numpy()
    for duration in [d for d in DURATIONS if d != BASELINE]:
        current = df.query('benchmark == @benchmark and duration_s == @duration')['energy_j'].to_numpy()
        test = stats.mannwhitneyu(current, base, alternative='two-sided')
        rows.append({'benchmark': benchmark, 'duration_s': duration, 'baseline_s': BASELINE,
                     'n_current': len(current), 'n_baseline': len(base),
                     'median_current_j': np.median(current), 'median_baseline_j': np.median(base),
                     'relative_median_change': np.median(current) / np.median(base) - 1,
                     'p_value': test.pvalue, 'cliffs_delta': cliffs_delta(current, base)})
rq0 = pd.DataFrame(rows)
rq0['p_adjusted_bh'] = bh(rq0['p_value'])
rq0['significant_bh'] = rq0['p_adjusted_bh'] < ALPHA
rq0.to_csv(OUT / 'table_rq0_baseline_comparisons.csv', index=False)
display(rq0.round(5))

In [ ]:
# RQ1: one-sided tests ask whether a candidate has larger deviations than 210 s.
rows = []
for benchmark in sorted(df['benchmark'].unique()):
    reference = df.query('benchmark == @benchmark and duration_s == @REFERENCE')['absolute_deviation_j'].to_numpy()
    reference_mean = reference.mean()
    reference_time = df.query('benchmark == @benchmark and duration_s == @REFERENCE')['total_time_s'].mean()
    for duration in [d for d in DURATIONS if d != REFERENCE]:
        current = df.query('benchmark == @benchmark and duration_s == @duration')['absolute_deviation_j'].to_numpy()
        test = stats.mannwhitneyu(current, reference, alternative='greater')
        rows.append({'benchmark': benchmark, 'duration_s': duration, 'reference_s': REFERENCE,
                     'mean_absolute_deviation_j': current.mean(), 'reference_mean_absolute_deviation_j': reference_mean,
                     'ratio_to_210s': current.mean() / reference_mean if reference_mean else np.nan,
                     'mean_total_time_s': df.query('benchmark == @benchmark and duration_s == @duration')['total_time_s'].mean(),
                     'reference_mean_total_time_s': reference_time, 'p_value': test.pvalue,
                     'cliffs_delta': cliffs_delta(current, reference)})
rq1 = pd.DataFrame(rows)
rq1['p_adjusted_bh'] = bh(rq1['p_value'])
rq1['not_significantly_larger'] = rq1['p_adjusted_bh'] >= ALPHA
rq1['within_10_percent'] = rq1['ratio_to_210s'] <= RATIO_LIMIT
rq1['suitable'] = rq1['not_significantly_larger'] & rq1['within_10_percent']
rq1.to_csv(OUT / 'table_rq1_tradeoff.csv', index=False)
recommendations = (rq1[rq1['suitable']].sort_values('duration_s').groupby('benchmark', as_index=False).first()[['benchmark','duration_s']].rename(columns={'duration_s':'recommended_duration_s'}))
recommendations.to_csv(OUT / 'table_rq1_recommendations.csv', index=False)
display(rq1.round(5))
display(recommendations)

Interpret p-values together with effect sizes and the descriptive tables. If the factorial assumptions are rejected, the notebook's pairwise Mann-Whitney results provide the planned non-parametric comparisons; do not describe them as evidence of causality beyond this randomized controlled setup.